# Amar Passport — AI Agent (Multi-Agent System with CrewAI)
**Assignment: Building a Virtual Consular Officer using CrewAI + Google Gemini**

This notebook implements a 3-agent CrewAI system that takes a user profile (Age, Profession, Urgency) and outputs a comprehensive **Passport Readiness Report** in both English and Bangla.

## Cell 1 — Install Dependencies

In [1]:
# Dependencies installed in .venv — run from the activated virtual environment
# pip install crewai crewai-tools duckduckgo-search ipykernel
print("✅ Dependencies already installed in .venv")

✅ Dependencies already installed in .venv


## Cell 2 — Imports & API Key Configuration

In [15]:
import os
import json
from IPython.display import Markdown, display
from dotenv import load_dotenv

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

# ─── Load API key from .env file ───────────────────────────────────────────────
load_dotenv()  # loads GEMINI_API_KEY and CREWAI_TELEMETRY_OPT_OUT from .env
# ───────────────────────────────────────────────────────────────────────────────

# gemini-2.5-flash-lite — separate daily quota pool from gemini-2.5-flash
llm = LLM(model="gemini/gemini-2.5-flash-lite")

print("✅ Imports done. LLM configured: gemini/gemini-2.5-flash-lite")
print(f"   GEMINI_API_KEY loaded: {'✅ yes' if os.getenv('GEMINI_API_KEY') else '❌ NOT FOUND — check .env'}")


✅ Imports done. LLM configured: gemini/gemini-2.5-flash-lite
   GEMINI_API_KEY loaded: ✅ yes


## Cell 3 — Local Fallback Database
This local DB is used as fallback when web scraping fails, and also as the authoritative 2026 fee structure.

In [3]:
LOCAL_DB = {
    "fees_2026": {
        "48_pages": {
            "5_years":  {"regular": 4025,  "express": 6325,  "super_express": 8625},
            "10_years": {"regular": 5750,  "express": 8050,  "super_express": 10350}
        },
        "64_pages": {
            "5_years":  {"regular": 6325,  "express": 8625,  "super_express": 12075},
            "10_years": {"regular": 8050,  "express": 10350, "super_express": 13800}
        }
    },
    "required_docs": {
        "adult":           ["NID Card", "Application Summary", "Payment Slip"],
        "minor_under_18":  ["Birth Registration (English)", "Parents NID", "3R Photo"],
        "government_staff":["NOC (No Objection Certificate)", "NID"]
    },
    "policy": {
        "validity_rules": {
            "under_18":  {"max_validity": 5, "id_type": "Birth Registration"},
            "18_to_65":  {"max_validity": 10, "id_type": "NID Card"},
            "over_65":   {"max_validity": 5, "id_type": "NID Card"}
        }
    }
}

print("✅ Local DB loaded successfully.")
print(f"   Fee categories: {list(LOCAL_DB['fees_2026'].keys())}")
print(f"   Doc categories: {list(LOCAL_DB['required_docs'].keys())}")

✅ Local DB loaded successfully.
   Fee categories: ['48_pages', '64_pages']
   Doc categories: ['adult', 'minor_under_18', 'government_staff']


## Cell 4 — Custom Tools (Web Scrape + Fallback)

In [4]:
@tool("Fetch Passport Policy")
def fetch_passport_policy(query: str) -> str:
    """
    Fetches current Bangladesh e-passport policy from the official portal
    (https://www.epassport.gov.bd). Falls back to local database if scraping fails.
    Input: a query string about passport policy.
    """
    try:
        from duckduckgo_search import DDGS
        with DDGS() as ddgs:
            results = list(ddgs.text(
                f"site:epassport.gov.bd {query} Bangladesh passport 2026",
                max_results=3
            ))
        if results:
            snippets = "\n".join([r.get("body", "") for r in results if r.get("body")])
            return f"[Web Result]\n{snippets}"
        raise ValueError("No results from web search.")
    except Exception as e:
        # ── FALLBACK to local DB ──
        policy = LOCAL_DB["policy"]["validity_rules"]
        fallback = (
            "[FALLBACK - Local DB] Bangladesh Passport Policy 2026:\n"
            f"  • Under 18  : max {policy['under_18']['max_validity']}-year validity, "
            f"ID required: {policy['under_18']['id_type']}\n"
            f"  • 18 to 65  : max {policy['18_to_65']['max_validity']}-year validity, "
            f"ID required: {policy['18_to_65']['id_type']}\n"
            f"  • Over 65   : max {policy['over_65']['max_validity']}-year validity, "
            f"ID required: {policy['over_65']['id_type']}\n"
            f"  (Fallback reason: {str(e)})"
        )
        return fallback


@tool("Calculate Passport Fee")
def calculate_fee(pages: int, validity_years: int, delivery: str) -> str:
    """
    Calculates the total BDT passport fee (VAT already included) based on:
    - pages: 48 or 64
    - validity_years: 5 or 10
    - delivery: 'regular', 'express', or 'super_express'
    Returns the fee amount in BDT as per 2026 official fee structure.
    """
    pages_key    = f"{pages}_pages"
    validity_key = f"{validity_years}_years"
    delivery_key = delivery.lower().replace(" ", "_")

    try:
        fee = LOCAL_DB["fees_2026"][pages_key][validity_key][delivery_key]
        return (
            f"Fee for {pages}-page, {validity_years}-year, {delivery} passport: "
            f"**{fee} BDT** (15% VAT included as per 2026 official fee structure)"
        )
    except KeyError:
        return (
            f"ERROR: Invalid combination — pages={pages}, validity={validity_years}y, "
            f"delivery={delivery}. Valid options: pages=[48,64], "
            f"validity=[5,10], delivery=[regular, express, super_express]."
        )


@tool("Get Required Documents")
def get_required_documents(age: int, profession: str, is_name_change: bool) -> str:
    """
    Returns a customized document checklist based on:
    - age: applicant's age (int)
    - profession: e.g., 'government', 'private', 'student', 'doctor'
    - is_name_change: True if applicant is changing their name on passport
    """
    docs = []
    profession_lower = profession.lower()

    if age < 18:
        docs.extend(LOCAL_DB["required_docs"]["minor_under_18"])
    else:
        docs.extend(LOCAL_DB["required_docs"]["adult"])

    govt_keywords = ["government", "govt", "public servant", "civil servant", 
                     "military", "police", "army", "bcs", "bank"]
    if any(kw in profession_lower for kw in govt_keywords):
        docs.extend(LOCAL_DB["required_docs"]["government_staff"])

    if is_name_change:
        docs.append("Marriage Certificate / Court Affidavit (for name change)")

    # Deduplicate while preserving order
    seen = set()
    unique_docs = []
    for d in docs:
        if d not in seen:
            seen.add(d)
            unique_docs.append(d)

    formatted = "\n".join([f"  {i+1}. {d}" for i, d in enumerate(unique_docs)])
    return f"Required Documents:\n{formatted}"


print("✅ Tools defined: fetch_passport_policy, calculate_fee, get_required_documents")

✅ Tools defined: fetch_passport_policy, calculate_fee, get_required_documents


## Cell 5 — Agent Definitions

In [5]:
# ─── Agent 1: The Policy Guardian ────────────────────────────────────────────
policy_guardian = Agent(
    role="Bangladesh Passport Policy Expert",
    goal=(
        "Determine the permitted passport validity (5 vs 10 years) and required "
        "identification document (NID vs Birth Registration) based on the applicant's "
        "age. Flag any policy violations immediately — especially if a minor (under 18) "
        "or senior (over 65) requests a 10-year passport."
    ),
    backstory=(
        "You are a veteran immigration officer at Agargaon Passport Office, Dhaka, "
        "with 15 years of experience enforcing Bangladesh passport regulations. "
        "You know every rule by heart: applicants under 18 CANNOT receive a 10-year "
        "passport — maximum is 5 years with Birth Registration (English version). "
        "Applicants over 65 also face the same 5-year restriction. "
        "Adults aged 18–65 are eligible for either 5 or 10 years using their NID. "
        "You always verify age against policy before proceeding."
    ),
    tools=[fetch_passport_policy],
    llm=llm,
    verbose=True
)

# ─── Agent 2: The Chancellor of the Exchequer ─────────────────────────────────
fee_calculator = Agent(
    role="Passport Financial Auditor",
    goal=(
        "Calculate the exact BDT fee (including 15% VAT) for the applicant's passport "
        "based on page count (48 or 64 pages), delivery speed (Regular / Express / "
        "Super Express), and validity. Use the 2026 official Bangladesh fee structure."
    ),
    backstory=(
        "You are the Chief Financial Officer of the Bangladesh Passport Department. "
        "You have memorized the complete 2026 fee schedule and always produce fees that "
        "include 15% VAT as mandated by the National Board of Revenue. "
        "You receive eligibility decisions from the Policy Guardian and translate them "
        "into exact costs. You never guess fees — you always use the official table."
    ),
    tools=[calculate_fee],
    llm=llm,
    verbose=True
)

# ─── Agent 3: The Document Architect ─────────────────────────────────────────
doc_architect = Agent(
    role="Passport Documentation Officer",
    goal=(
        "Generate a complete, customized document checklist for the applicant. "
        "Account for special cases: GO/NOC for government employees, "
        "Marriage Certificate for name changes, Parent's NID + 3R photo for minors. "
        "Output the final Passport Readiness Report as a formatted Markdown table in English."
    ),
    backstory=(
        "You are the Head of Documentation at the Bangladesh Immigration Authority. "
        "Your responsibility is ensuring applicants arrive with every single required document. "
        "You tailor checklists precisely — a government employee needs a No Objection "
        "Certificate (NOC) from their employer; a minor needs English Birth Registration "
        "and both parents' NIDs; anyone changing their name needs a Marriage Certificate "
        "or court affidavit. You compile all findings into a clear, readable Markdown table."
    ),
    tools=[get_required_documents],
    llm=llm,
    verbose=True
)

print("✅ Agents defined: Policy Guardian, Fee Calculator, Document Architect")

✅ Agents defined: Policy Guardian, Fee Calculator, Document Architect


## Cell 6 — Task Definitions

In [6]:
# ─── Task 1: Eligibility Check ────────────────────────────────────────────────
task_eligibility = Task(
    description=(
        "Analyze the following applicant profile and determine passport eligibility:\n"
        "  - Age: {age}\n"
        "  - Profession: {profession}\n"
        "  - Requested validity: {requested_validity} years\n"
        "  - Urgency: {urgency}\n"
        "  - Location: {location}\n\n"
        "Steps:\n"
        "1. Use the 'Fetch Passport Policy' tool to retrieve current Bangladesh passport rules.\n"
        "2. Determine the MAXIMUM permitted validity based on age (under 18 → max 5 years; "
        "18–65 → max 10 years; over 65 → max 5 years).\n"
        "3. Determine the required ID type (under 18 → Birth Registration; 18+ → NID).\n"
        "4. If the requested validity exceeds the permitted maximum, output a POLICY VIOLATION "
        "flag in bold: **⚠️ POLICY VIOLATION: [reason]**\n"
        "5. Recommend the correct delivery type based on urgency: "
        "'2 weeks or less' → Express; 'very urgent/next day' → Super Express; "
        "otherwise → Regular.\n"
    ),
    expected_output=(
        "A structured eligibility report with: permitted validity (years), recommended "
        "delivery type, required ID type, and any policy violation flags."
    ),
    agent=policy_guardian
)

# ─── Task 2: Fee Calculation (receives Task 1 output as context) ──────────────
task_fee = Task(
    description=(
        "Using the eligibility decision from the Policy Guardian, calculate the exact fee:\n"
        "  - Page count: {pages} pages\n"
        "  - Validity: use the PERMITTED validity from Task 1 (not necessarily what was requested)\n"
        "  - Delivery: use the recommended delivery type from Task 1\n\n"
        "Use the 'Calculate Passport Fee' tool with the correct parameters. "
        "Report the total fee in BDT, confirming that 15% VAT is included."
    ),
    expected_output=(
        "The exact total fee in BDT including 15% VAT, with breakdown of page count, "
        "validity, and delivery type chosen."
    ),
    agent=fee_calculator,
    context=[task_eligibility]
)

# ─── Task 3: Document Checklist + Final Report ────────────────────────────────
task_checklist = Task(
    description=(
        "Using the eligibility and fee information from previous tasks, generate the final report:\n"
        "  - Age: {age}\n"
        "  - Profession: {profession}\n"
        "  - Is name change: {is_name_change}\n\n"
        "Steps:\n"
        "1. Use the 'Get Required Documents' tool to get the customized checklist.\n"
        "2. Compile ALL findings into a comprehensive Passport Readiness Report.\n"
        "3. Format the final output as a Markdown table with these columns:\n"
        "   | Field | Details |\n"
        "   Include rows for: Applicant Age, Permitted Validity, Delivery Type, "
        "Total Fee (BDT), Required ID, Documents Needed, Policy Notes.\n"
        "4. After the English Markdown table, also provide a brief summary in Bangla "
        "(Bengali script) covering validity, fee, and doc requirements."
    ),
    expected_output=(
        "A complete Passport Readiness Report formatted as a Markdown table in English, "
        "followed by a brief summary in Bangla (Bengali script)."
    ),
    agent=doc_architect,
    context=[task_eligibility, task_fee]
)

print("✅ Tasks defined: task_eligibility, task_fee, task_checklist")

✅ Tasks defined: task_eligibility, task_fee, task_checklist


## Cell 7 — Crew Assembly

In [7]:
passport_crew = Crew(
    agents=[policy_guardian, fee_calculator, doc_architect],
    tasks=[task_eligibility, task_fee, task_checklist],
    process=Process.sequential,
    verbose=True
)

print("✅ Crew assembled with sequential process.")
print(f"   Agents: {[a.role for a in passport_crew.agents]}")
print(f"   Tasks : {len(passport_crew.tasks)} tasks")

✅ Crew assembled with sequential process.
   Agents: ['Bangladesh Passport Policy Expert', 'Passport Financial Auditor', 'Passport Documentation Officer']
   Tasks : 3 tasks


In [8]:
# ── Original task description templates (hardcoded, since kickoff fills them in-place) ──
TASK_ELIG_DESC = (
    "Analyze the following applicant profile and determine passport eligibility:\n"
    "  - Age: {age}\n"
    "  - Profession: {profession}\n"
    "  - Requested validity: {requested_validity} years\n"
    "  - Urgency: {urgency}\n"
    "  - Location: {location}\n\n"
    "Steps:\n"
    "1. Use the 'Fetch Passport Policy' tool to retrieve current Bangladesh passport rules.\n"
    "2. Determine the MAXIMUM permitted validity based on age (under 18 → max 5 years; "
    "18–65 → max 10 years; over 65 → max 5 years).\n"
    "3. Determine the required ID type (under 18 → Birth Registration; 18+ → NID).\n"
    "4. If the requested validity exceeds the permitted maximum, output a POLICY VIOLATION "
    "flag in bold: **⚠️ POLICY VIOLATION: [reason]**\n"
    "5. Recommend the correct delivery type based on urgency: "
    "'2 weeks or less' → Express; 'very urgent/next day' → Super Express; "
    "otherwise → Regular.\n"
)

TASK_ELIG_EXPECTED = (
    "A structured eligibility report with: permitted validity (years), recommended "
    "delivery type, required ID type, and any policy violation flags."
)

TASK_FEE_DESC = (
    "Using the eligibility decision from the Policy Guardian, calculate the exact fee:\n"
    "  - Page count: {pages} pages\n"
    "  - Validity: use the PERMITTED validity from Task 1 (not necessarily what was requested)\n"
    "  - Delivery: use the recommended delivery type from Task 1\n\n"
    "Use the 'Calculate Passport Fee' tool with the correct parameters. "
    "Report the total fee in BDT, confirming that 15% VAT is included."
)

TASK_FEE_EXPECTED = (
    "The exact total fee in BDT including 15% VAT, with breakdown of page count, "
    "validity, and delivery type chosen."
)

TASK_CHECK_DESC = (
    "Using the eligibility and fee information from previous tasks, generate the final report:\n"
    "  - Age: {age}\n"
    "  - Profession: {profession}\n"
    "  - Is name change: {is_name_change}\n\n"
    "Steps:\n"
    "1. Use the 'Get Required Documents' tool to get the customized checklist.\n"
    "2. Compile ALL findings into a comprehensive Passport Readiness Report.\n"
    "3. Format the final output as a Markdown table with these columns:\n"
    "   | Field | Details |\n"
    "   Include rows for: Applicant Age, Permitted Validity, Delivery Type, "
    "Total Fee (BDT), Required ID, Documents Needed, Policy Notes.\n"
    "4. After the English Markdown table, also provide a brief summary in Bangla "
    "(Bengali script) covering validity, fee, and doc requirements."
)

TASK_CHECK_EXPECTED = (
    "A complete Passport Readiness Report formatted as a Markdown table in English, "
    "followed by a brief summary in Bangla (Bengali script)."
)

print("✅ Original task description templates restored as constants.")
print("   These will be used by Scenarios 2 & 3 to ensure correct input substitution.")


✅ Original task description templates restored as constants.
   These will be used by Scenarios 2 & 3 to ensure correct input substitution.


## Cell 8 — Scenario 1: Primary Example (Adult, Private Sector, Urgent)

In [9]:
# ── Example Input Scenario from Assignment ────────────────────────────────────
user_profile_1 = {
    "age": 24,
    "profession": "private sector employee",
    "requested_validity": 10,
    "urgency": "urgent — business trip in 2 weeks",
    "pages": 64,
    "location": "Dhaka",
    "is_name_change": False
}

print("=" * 60)
print("SCENARIO 1: 24-year-old private sector employee")
print("  64-page passport, urgent (business trip in 2 weeks)")
print("  Has NID, lives in Dhaka")
print("=" * 60)

result_1 = passport_crew.kickoff(inputs=user_profile_1)

print("\n" + "=" * 60)
print("FINAL PASSPORT READINESS REPORT — SCENARIO 1")
print("=" * 60)

SCENARIO 1: 24-year-old private sector employee
  64-page passport, urgent (business trip in 2 weeks)
  Has NID, lives in Dhaka


╭──────────────────────────────────────────────── Yanked Version ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Version 1.10.0 has been yanked from PyPI.                                                                      │
│  Reason: miss behaving when running on crewai AMP                                                               │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6ce115b8-85e9-4c5b-a30e-623e8590c9bd                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the following applicant profile and determine passport eligibility:                              │
│    - Age: 24                                                                                                    │
│    - Profession: private sector employee                                                                        │
│    - Requested validity: 10 years                                                                               │
│    - Urgency: urgent — business trip in 2 weeks                                                                 │
│    - Location: Dhaka                                                                                            │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the 'Fetch Passport Policy' tool to retrieve current Bangladesh passport rules.                         │
│  2. Determine the MAXIMUM permitted validity based on age (under 18 → max 5 years; 18–65 → max 10 years; over   │
│  65 → max 5 years).                                                                                             │
│  3. Determine the required ID type (under 18 → Birth Registration; 18+ → NID).                                  │
│  4. If the requested validity exceeds the permitted maximum, output a POLICY VIOLATION flag in bold: **⚠️       │
│  POLICY VIOLATION: [reason]**                                                                                   │
│  5. Recommend the correct delivery type based on urgency: '2 weeks or less' → Express; 'very urgent/next day'   │
│  → Super Express; otherwise → Regular.                                                                          │
│                                                                                                                 │
│  ID: f4ef0ce7-04c6-4323-8eeb-f982a0f52e4e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Bangladesh Passport Policy Expert                                                                       │
│                                                                                                                 │
│  Task: Analyze the following applicant profile and determine passport eligibility:                              │
│    - Age: 24                                                                                                    │
│    - Profession: private sector employee                                                                        │
│    - Requested validity: 10 years                                                                               │
│    - Urgency: urgent — business trip in 2 weeks                                                                 │
│    - Location: Dhaka                                                                                            │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the 'Fetch Passport Policy' tool to retrieve current Bangladesh passport rules.                         │
│  2. Determine the MAXIMUM permitted validity based on age (under 18 → max 5 years; 18–65 → max 10 years; over   │
│  65 → max 5 years).                                                                                             │
│  3. Determine the required ID type (under 18 → Birth Registration; 18+ → NID).                                  │
│  4. If the requested validity exceeds the permitted maximum, output a POLICY VIOLATION flag in bold: **⚠️       │
│  POLICY VIOLATION: [reason]**                                                                                   │
│  5. Recommend the correct delivery type based on urgency: '2 weeks or less' → Express; 'very urgent/next day'   │
│  → Super Express; otherwise → Regular.                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Passport Documentation Officer                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  | Field | Details |                                                                                            │
│  |---|---|                                                                                                      │
│  | Applicant Age | 24 |                                                                                         │
│  | Permitted Validity | 10 years |                                                                              │
│  | Delivery Type | Express |                                                                                    │
│  | Total Fee (BDT) | 10350 |                                                                                    │
│  | Required ID | NID |                                                                                          │
│  | Documents Needed | 1. NID Card<br>2. Application Summary<br>3. Payment Slip |                                │
│  | Policy Notes | None |                                                                                        │
│                                                                                                                 │
│  **প্রয়োজনীয় তথ্যের সারাংশ:**                                                                                           │
│                                                                                                                 │
│  *   **বৈধতা:** ১০ বছর                                                                                            │
│  *   **ফি:** ১০,৩৫০ টাকা                                                                                           │
│  *   **প্রয়োজনীয় নথি:** এনআইডি কার্ড, আবেদন সারাংশ, পেমেন্ট স্লিপ।                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the eligibility and fee information from previous tasks, generate the final report:                │
│    - Age: 24                                                                                                    │
│    - Profession: private sector employee                                                                        │
│    - Is name change: False                                                                                      │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the 'Get Required Documents' tool to get the customized checklist.                                      │
│  2. Compile ALL findings into a comprehensive Passport Readiness Report.                                        │
│  3. Format the final output as a Markdown table with these columns:                                             │
│     | Field | Details |                                                                                         │
│     Include rows for: Applicant Age, Permitted Validity, Delivery Type, Total Fee (BDT), Required ID,           │
│  Documents Needed, Policy Notes.                                                                                │
│  4. After the English Markdown table, also provide a brief summary in Bangla (Bengali script) covering          │
│  validity, fee, and doc requirements.                                                                           │
│  Agent: Passport Documentation Officer                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6ce115b8-85e9-4c5b-a30e-623e8590c9bd                                                                       │
│  Final Output: | Field | Details |                                                                              │
│  |---|---|                                                                                                      │
│  | Applicant Age | 24 |                                                                                         │
│  | Permitted Validity | 10 years |                                                                              │
│  | Delivery Type | Express |                                                                                    │
│  | Total Fee (BDT) | 10350 |                                                                                    │
│  | Required ID | NID |                                                                                          │
│  | Documents Needed | 1. NID Card<br>2. Application Summary<br>3. Payment Slip |                                │
│  | Policy Notes | None |                                                                                        │
│                                                                                                                 │
│  **প্রয়োজনীয় তথ্যের সারাংশ:**                                                                                           │
│                                                                                                                 │
│  *   **বৈধতা:** ১০ বছর                                                                                            │
│  *   **ফি:** ১০,৩৫০ টাকা                                                                                           │
│  *   **প্রয়োজনীয় নথি:** এনআইডি কার্ড, আবেদন সারাংশ, পেমেন্ট স্লিপ।                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL PASSPORT READINESS REPORT — SCENARIO 1


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [10]:
# Display the rendered Markdown report
display(Markdown(result_1.raw))

| Field | Details |
|---|---|
| Applicant Age | 24 |
| Permitted Validity | 10 years |
| Delivery Type | Express |
| Total Fee (BDT) | 10350 |
| Required ID | NID |
| Documents Needed | 1. NID Card<br>2. Application Summary<br>3. Payment Slip |
| Policy Notes | None |

**প্রয়োজনীয় তথ্যের সারাংশ:**

*   **বৈধতা:** ১০ বছর
*   **ফি:** ১০,৩৫০ টাকা
*   **প্রয়োজনীয় নথি:** এনআইডি কার্ড, আবেদন সারাংশ, পেমেন্ট স্লিপ।

## Cell 9 — Scenario 2: Error Handling — Minor Requesting 10-Year Passport
This demonstrates the Policy Guardian's **POLICY VIOLATION** flag.

In [11]:
import time

# Re-create tasks for scenario 2 using original templates (not the filled-in versions)
task_elig_2 = Task(
    description=TASK_ELIG_DESC,
    expected_output=TASK_ELIG_EXPECTED,
    agent=policy_guardian
)
task_fee_2 = Task(
    description=TASK_FEE_DESC,
    expected_output=TASK_FEE_EXPECTED,
    agent=fee_calculator,
    context=[task_elig_2]
)
task_check_2 = Task(
    description=TASK_CHECK_DESC,
    expected_output=TASK_CHECK_EXPECTED,
    agent=doc_architect,
    context=[task_elig_2, task_fee_2]
)

crew_2 = Crew(
    agents=[policy_guardian, fee_calculator, doc_architect],
    tasks=[task_elig_2, task_fee_2, task_check_2],
    process=Process.sequential,
    verbose=True
)

user_profile_2 = {
    "age": 15,
    "profession": "student",
    "requested_validity": 10,   # ← POLICY VIOLATION: minor cannot get 10-year
    "urgency": "regular, no rush",
    "pages": 48,
    "location": "Chittagong",
    "is_name_change": False
}

print("⏳ Waiting 90s for API rate-limit reset (free tier: 5 RPM)...")
time.sleep(90)

print("=" * 60)
print("SCENARIO 2: 15-year-old student requesting 10-year passport")
print("  ⚠️  This should trigger a POLICY VIOLATION flag!")
print("=" * 60)

result_2 = crew_2.kickoff(inputs=user_profile_2)

print("\n" + "=" * 60)
print("FINAL PASSPORT READINESS REPORT — SCENARIO 2 (Minor)")
print("=" * 60)


⏳ Waiting 90s for API rate-limit reset (free tier: 5 RPM)...
SCENARIO 2: 15-year-old student requesting 10-year passport
  ⚠️  This should trigger a POLICY VIOLATION flag!


╭──────────────────────────────────────────────── Yanked Version ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Version 1.10.0 has been yanked from PyPI.                                                                      │
│  Reason: miss behaving when running on crewai AMP                                                               │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 26c40852-63d8-4bb8-b71b-8c7906425f15                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/home/meow/Assignment_Building_Amar Passport_AI Agent/.venv/lib/python3.12/site-packages/crewai/crew.py:1289: DeprecationWarning: deprecated
  and hasattr(agent, "multimodal")
/home/meow/Assignment_Building_Amar Passport_AI Agent/.venv/lib/python3.12/site-packages/crewai/crew.py:1290: DeprecationWarning: deprecated
  and getattr(agent, "multimodal", False)


╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the following applicant profile and determine passport eligibility:                              │
│    - Age: 15                                                                                                    │
│    - Profession: student                                                                                        │
│    - Requested validity: 10 years                                                                               │
│    - Urgency: regular, no rush                                                                                  │
│    - Location: Chittagong                                                                                       │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the 'Fetch Passport Policy' tool to retrieve current Bangladesh passport rules.                         │
│  2. Determine the MAXIMUM permitted validity based on age (under 18 → max 5 years; 18–65 → max 10 years; over   │
│  65 → max 5 years).                                                                                             │
│  3. Determine the required ID type (under 18 → Birth Registration; 18+ → NID).                                  │
│  4. If the requested validity exceeds the permitted maximum, output a POLICY VIOLATION flag in bold: **⚠️       │
│  POLICY VIOLATION: [reason]**                                                                                   │
│  5. Recommend the correct delivery type based on urgency: '2 weeks or less' → Express; 'very urgent/next day'   │
│  → Super Express; otherwise → Regular.                                                                          │
│                                                                                                                 │
│  ID: 2f449a33-8bd4-4779-bcf8-5390ebab285d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Passport Documentation Officer                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  | Field | Details |                                                                                            │
│  |---|---|                                                                                                      │
│  | Applicant Age | 15 |                                                                                         │
│  | Permitted Validity | 5 years |                                                                               │
│  | Delivery Type | Regular |                                                                                    │
│  | Total Fee (BDT) | 4025 |                                                                                     │
│  | Required ID | Birth Registration |                                                                           │
│  | Documents Needed | 1. Birth Registration (English)<br>2. Parents NID<br>3. 3R Photo |                        │
│  | Policy Notes | ⚠️ POLICY VIOLATION: Applicant requested 10-year passport, but is under 18, and thus          │
│  eligible for a maximum of 5 years. |                                                                           │
│                                                                                                                 │
│  **প্রয়োজনীয় তথ্যের সারাংশ:**                                                                                           │
│                                                                                                                 │
│  *   **বৈধতা:** ৫ বছর                                                                                             │
│  *   **ফি:** ৪,০২৫ টাকা                                                                                            │
│  *   **প্রয়োজনীয় নথি:** ইংরেজি বার্থ রেজিস্ট্রেশন, পিতামাতার এনআইডি, ৩আর ছবি।                                                       │
│  *   **নীতি দ্রষ্টব্য:** আবেদনকারী ১০ বছরের পাসপোর্ট চেয়েছেন, কিন্তু ১৮ বছরের কম বয়সী হওয়ায় সর্বাধিক ৫ বছরের জন্য যোগ্য।                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the eligibility and fee information from previous tasks, generate the final report:                │
│    - Age: 15                                                                                                    │
│    - Profession: student                                                                                        │
│    - Is name change: False                                                                                      │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the 'Get Required Documents' tool to get the customized checklist.                                      │
│  2. Compile ALL findings into a comprehensive Passport Readiness Report.                                        │
│  3. Format the final output as a Markdown table with these columns:                                             │
│     | Field | Details |                                                                                         │
│     Include rows for: Applicant Age, Permitted Validity, Delivery Type, Total Fee (BDT), Required ID,           │
│  Documents Needed, Policy Notes.                                                                                │
│  4. After the English Markdown table, also provide a brief summary in Bangla (Bengali script) covering          │
│  validity, fee, and doc requirements.                                                                           │
│  Agent: Passport Documentation Officer                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 26c40852-63d8-4bb8-b71b-8c7906425f15                                                                       │
│  Final Output: | Field | Details |                                                                              │
│  |---|---|                                                                                                      │
│  | Applicant Age | 15 |                                                                                         │
│  | Permitted Validity | 5 years |                                                                               │
│  | Delivery Type | Regular |                                                                                    │
│  | Total Fee (BDT) | 4025 |                                                                                     │
│  | Required ID | Birth Registration |                                                                           │
│  | Documents Needed | 1. Birth Registration (English)<br>2. Parents NID<br>3. 3R Photo |                        │
│  | Policy Notes | ⚠️ POLICY VIOLATION: Applicant requested 10-year passport, but is under 18, and thus          │
│  eligible for a maximum of 5 years. |                                                                           │
│                                                                                                                 │
│  **প্রয়োজনীয় তথ্যের সারাংশ:**                                                                                           │
│                                                                                                                 │
│  *   **বৈধতা:** ৫ বছর                                                                                             │
│  *   **ফি:** ৪,০২৫ টাকা                                                                                            │
│  *   **প্রয়োজনীয় নথি:** ইংরেজি বার্থ রেজিস্ট্রেশন, পিতামাতার এনআইডি, ৩আর ছবি।                                                       │
│  *   **নীতি দ্রষ্টব্য:** আবেদনকারী ১০ বছরের পাসপোর্ট চেয়েছেন, কিন্তু ১৮ বছরের কম বয়সী হওয়ায় সর্বাধিক ৫ বছরের জন্য যোগ্য।                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL PASSPORT READINESS REPORT — SCENARIO 2 (Minor)


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [12]:
display(Markdown(result_2.raw))

| Field | Details |
|---|---|
| Applicant Age | 15 |
| Permitted Validity | 5 years |
| Delivery Type | Regular |
| Total Fee (BDT) | 4025 |
| Required ID | Birth Registration |
| Documents Needed | 1. Birth Registration (English)<br>2. Parents NID<br>3. 3R Photo |
| Policy Notes | ⚠️ POLICY VIOLATION: Applicant requested 10-year passport, but is under 18, and thus eligible for a maximum of 5 years. |

**প্রয়োজনীয় তথ্যের সারাংশ:**

*   **বৈধতা:** ৫ বছর
*   **ফি:** ৪,০২৫ টাকা
*   **প্রয়োজনীয় নথি:** ইংরেজি বার্থ রেজিস্ট্রেশন, পিতামাতার এনআইডি, ৩আর ছবি।
*   **নীতি দ্রষ্টব্য:** আবেদনকারী ১০ বছরের পাসপোর্ট চেয়েছেন, কিন্তু ১৮ বছরের কম বয়সী হওয়ায় সর্বাধিক ৫ বছরের জন্য যোগ্য।

## Cell 10 — Scenario 3: Government Employee with Name Change

In [13]:
task_elig_3 = Task(
    description=TASK_ELIG_DESC,
    expected_output=TASK_ELIG_EXPECTED,
    agent=policy_guardian
)
task_fee_3 = Task(
    description=TASK_FEE_DESC,
    expected_output=TASK_FEE_EXPECTED,
    agent=fee_calculator,
    context=[task_elig_3]
)
task_check_3 = Task(
    description=TASK_CHECK_DESC,
    expected_output=TASK_CHECK_EXPECTED,
    agent=doc_architect,
    context=[task_elig_3, task_fee_3]
)

crew_3 = Crew(
    agents=[policy_guardian, fee_calculator, doc_architect],
    tasks=[task_elig_3, task_fee_3, task_check_3],
    process=Process.sequential,
    verbose=True
)

user_profile_3 = {
    "age": 35,
    "profession": "government officer (BCS cadre)",
    "requested_validity": 10,
    "urgency": "regular, no rush",
    "pages": 48,
    "location": "Dhaka",
    "is_name_change": True    # ← requires Marriage Certificate
}

print("⏳ Waiting 90s for API rate-limit reset (free tier: 5 RPM)...")
time.sleep(90)

print("=" * 60)
print("SCENARIO 3: 35-year-old Government Officer (BCS), name change")
print("  Should require: NOC + NID + Marriage Certificate")
print("=" * 60)

result_3 = crew_3.kickoff(inputs=user_profile_3)

print("\n" + "=" * 60)
print("FINAL PASSPORT READINESS REPORT — SCENARIO 3 (Govt + Name Change)")
print("=" * 60)


⏳ Waiting 90s for API rate-limit reset (free tier: 5 RPM)...
SCENARIO 3: 35-year-old Government Officer (BCS), name change
  Should require: NOC + NID + Marriage Certificate


╭──────────────────────────────────────────────── Yanked Version ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Version 1.10.0 has been yanked from PyPI.                                                                      │
│  Reason: miss behaving when running on crewai AMP                                                               │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 86c1f523-f8a8-490c-a1b4-25421797a035                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

/home/meow/Assignment_Building_Amar Passport_AI Agent/.venv/lib/python3.12/site-packages/crewai/crew.py:1289: DeprecationWarning: deprecated
  and hasattr(agent, "multimodal")
/home/meow/Assignment_Building_Amar Passport_AI Agent/.venv/lib/python3.12/site-packages/crewai/crew.py:1290: DeprecationWarning: deprecated
  and getattr(agent, "multimodal", False)


╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Analyze the following applicant profile and determine passport eligibility:                              │
│    - Age: 35                                                                                                    │
│    - Profession: government officer (BCS cadre)                                                                 │
│    - Requested validity: 10 years                                                                               │
│    - Urgency: regular, no rush                                                                                  │
│    - Location: Dhaka                                                                                            │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the 'Fetch Passport Policy' tool to retrieve current Bangladesh passport rules.                         │
│  2. Determine the MAXIMUM permitted validity based on age (under 18 → max 5 years; 18–65 → max 10 years; over   │
│  65 → max 5 years).                                                                                             │
│  3. Determine the required ID type (under 18 → Birth Registration; 18+ → NID).                                  │
│  4. If the requested validity exceeds the permitted maximum, output a POLICY VIOLATION flag in bold: **⚠️       │
│  POLICY VIOLATION: [reason]**                                                                                   │
│  5. Recommend the correct delivery type based on urgency: '2 weeks or less' → Express; 'very urgent/next day'   │
│  → Super Express; otherwise → Regular.                                                                          │
│                                                                                                                 │
│  ID: dd822bcb-1a4b-40ab-bcf2-e37a09bf187f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Passport Documentation Officer                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  | Field | Details |                                                                                            │
│  |---|---|                                                                                                      │
│  | Applicant Age | 35 |                                                                                         │
│  | Permitted Validity | 10 years |                                                                              │
│  | Delivery Type | Regular |                                                                                    │
│  | Total Fee (BDT) | 5750 |                                                                                     │
│  | Required ID | NID |                                                                                          │
│  | Documents Needed | 1. NID Card<br>2. Application Summary<br>3. Payment Slip<br>4. NOC (No Objection          │
│  Certificate)<br>5. NID<br>6. Marriage Certificate / Court Affidavit (for name change) |                        │
│  | Policy Notes | None |                                                                                        │
│                                                                                                                 │
│  **প্রয়োজনীয় তথ্যের সারাংশ:**                                                                                           │
│                                                                                                                 │
│  *   **বৈধতা:** ১০ বছর                                                                                            │
│  *   **ফি:** ৫৭৫০ টাকা                                                                                             │
│  *   **প্রয়োজনীয় নথি:** এনআইডি কার্ড, আবেদন সারাংশ, পেমেন্ট স্লিপ, এনওসি (অনাপত্তি সনদ), এনআইডি, বিবাহ সনদ/আদালতের হলফনামা (নাম পরিবর্তনের     │
│  জন্য)।                                                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the eligibility and fee information from previous tasks, generate the final report:                │
│    - Age: 35                                                                                                    │
│    - Profession: government officer (BCS cadre)                                                                 │
│    - Is name change: True                                                                                       │
│                                                                                                                 │
│  Steps:                                                                                                         │
│  1. Use the 'Get Required Documents' tool to get the customized checklist.                                      │
│  2. Compile ALL findings into a comprehensive Passport Readiness Report.                                        │
│  3. Format the final output as a Markdown table with these columns:                                             │
│     | Field | Details |                                                                                         │
│     Include rows for: Applicant Age, Permitted Validity, Delivery Type, Total Fee (BDT), Required ID,           │
│  Documents Needed, Policy Notes.                                                                                │
│  4. After the English Markdown table, also provide a brief summary in Bangla (Bengali script) covering          │
│  validity, fee, and doc requirements.                                                                           │
│  Agent: Passport Documentation Officer                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 86c1f523-f8a8-490c-a1b4-25421797a035                                                                       │
│  Final Output: | Field | Details |                                                                              │
│  |---|---|                                                                                                      │
│  | Applicant Age | 35 |                                                                                         │
│  | Permitted Validity | 10 years |                                                                              │
│  | Delivery Type | Regular |                                                                                    │
│  | Total Fee (BDT) | 5750 |                                                                                     │
│  | Required ID | NID |                                                                                          │
│  | Documents Needed | 1. NID Card<br>2. Application Summary<br>3. Payment Slip<br>4. NOC (No Objection          │
│  Certificate)<br>5. NID<br>6. Marriage Certificate / Court Affidavit (for name change) |                        │
│  | Policy Notes | None |                                                                                        │
│                                                                                                                 │
│  **প্রয়োজনীয় তথ্যের সারাংশ:**                                                                                           │
│                                                                                                                 │
│  *   **বৈধতা:** ১০ বছর                                                                                            │
│  *   **ফি:** ৫৭৫০ টাকা                                                                                             │
│  *   **প্রয়োজনীয় নথি:** এনআইডি কার্ড, আবেদন সারাংশ, পেমেন্ট স্লিপ, এনওসি (অনাপত্তি সনদ), এনআইডি, বিবাহ সনদ/আদালতের হলফনামা (নাম পরিবর্তনের     │
│  জন্য)।                                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL PASSPORT READINESS REPORT — SCENARIO 3 (Govt + Name Change)


╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [14]:
display(Markdown(result_3.raw))

| Field | Details |
|---|---|
| Applicant Age | 35 |
| Permitted Validity | 10 years |
| Delivery Type | Regular |
| Total Fee (BDT) | 5750 |
| Required ID | NID |
| Documents Needed | 1. NID Card<br>2. Application Summary<br>3. Payment Slip<br>4. NOC (No Objection Certificate)<br>5. NID<br>6. Marriage Certificate / Court Affidavit (for name change) |
| Policy Notes | None |

**প্রয়োজনীয় তথ্যের সারাংশ:**

*   **বৈধতা:** ১০ বছর
*   **ফি:** ৫৭৫০ টাকা
*   **প্রয়োজনীয় নথি:** এনআইডি কার্ড, আবেদন সারাংশ, পেমেন্ট স্লিপ, এনওসি (অনাপত্তি সনদ), এনআইডি, বিবাহ সনদ/আদালতের হলফনামা (নাম পরিবর্তনের জন্য)।